# ClusterBP v2, stage 2: comparison with mixture of experts

The reviewer's question on "personalized fusion": does personalising how experts are mixed help over global mixing?
In `clusterbp_v2.ipynb` per-cluster blending (ClusterBP) gave no gain. Here we test whether a **learned** gate helps.

The experts are the fold models already trained in `clusterbp_v2.ipynb`: **GRU** (last 20 measurements) and **XGB_full** (BP features + event history).
Folds, inner validation and phenotypes are the same; test patients take no part in any training.

| Model | What it is | Counterpart in the literature |
|---|---|---|
| `Stack_global` | LR on the logits of the two experts | standard stacking |
| `Gate_pheno` | LR on the experts' logits and their products with soft GDTW phenotype membership | learnable version of ClusterBP |
| `MoE_gate` | GRU weight = MLP(patient and phenotype features), p = w·p_GRU + (1−w)·p_XGB | gate on "presentation data", Hurley et al., ACM HEALTH 2023 |
| `ResidualMoE` | base GRU for all + an XGBoost expert per phenotype trained on the residual (base_margin = GRU logit) | Lee & Hauskrecht, AIME 2022 |
| `Residual_global` | the same with one expert for all | control for `ResidualMoE` |

Gates (`Stack_global`, `Gate_pheno`, `MoE_gate`) are trained on inner validation: on the training part the experts' predictions are in-sample and would be over-optimistic.
Residual experts are trained on the training part with early stopping on validation; the base GRU is in-sample there, a limitation of the Lee & Hauskrecht scheme.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score
from xgboost import XGBClassifier

from clusterbp_lib import (STATIC_COLS, CACHE_DIR, RESULTS_DIR, SEED, N_FOLDS, N_CLUSTERS, FEATURE_SETS,
                           load_cached, fold_masks, add_phenotype_columns, best_threshold,
                           load_gru, load_xgb, gru_logits, sigmoid, logit, boot_eval, ci)

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())

MOE_TARGETS = ['A', 'B', 'OLD']
GATE_COLS = STATIC_COLS + ['n_meas', 'n_days',
             'sbp_mean', 'sbp_std', 'sbp_arv', 'sbp_slope', 'pd0', 'pd1', 'pd2']
N_BOOT = 1000

win, X_seq, seq_len, pheno_win, oof_v2 = load_cached()
print(f'Windows: {len(win):,}, patients: {win.pid.nunique():,}')

Windows: 420,355, patients: 9,413


In [2]:
def memberships(dist, temp=1.0):
    """Soft phenotype membership: softmax(−GDTW distance)."""
    d = np.where(np.isnan(dist), np.nanmax(dist), dist)
    z = -(d - d.min(1, keepdims=True)) / temp
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_gate_lr(Z_va, y_va):
    sc = StandardScaler().fit(Z_va)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(Z_va), y_va)
    return lambda Z: lr.predict_proba(sc.transform(Z))[:, 1]


class Gate(nn.Module):
    def __init__(self, n_in, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, z, pg, px):
        w = torch.sigmoid(self.net(z).squeeze(-1))
        return w * pg + (1 - w) * px, w


def fit_moe_gate(Z, pg, px, y, seed, epochs=200, patience=10):
    """The gate is trained on inner validation; 20% of its patients are used for early stopping."""
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    med = np.nanmedian(Z, 0)
    Z = np.where(np.isnan(Z), med, Z)
    mu, sd = Z.mean(0), Z.std(0) + 1e-6
    Zs = ((Z - mu) / sd).astype(np.float32)
    idx = rng.permutation(len(y))
    n_es = len(y) // 5
    es, tr = idx[:n_es], idx[n_es:]
    T = lambda a: torch.from_numpy(np.asarray(a, dtype=np.float32))
    model = Gate(Z.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, best_state, bad = np.inf, None, 0
    for _ in range(epochs):
        model.train()
        for b in np.array_split(rng.permutation(tr), max(1, len(tr) // 2048)):
            opt.zero_grad()
            p, _ = model(T(Zs[b]), T(pg[b]), T(px[b]))
            loss = nn.functional.binary_cross_entropy(p.clamp(1e-6, 1 - 1e-6), T(y[b]))
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            p, _ = model(T(Zs[es]), T(pg[es]), T(px[es]))
            l = nn.functional.binary_cross_entropy(p.clamp(1e-6, 1 - 1e-6), T(y[es])).item()
        if l < best - 1e-5:
            best, bad = l, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)

    def predict(Zn, pgn, pxn):
        Zn = (np.where(np.isnan(Zn), med, Zn) - mu) / sd
        with torch.no_grad():
            p, w = model(T(Zn), T(pgn), T(pxn))
        return p.numpy(), w.numpy()
    return predict


def fit_residual(X, base, y, tr, va, seed):
    """XGBoost expert trained on the residual relative to the base model (base_margin = base logit)."""
    m = XGBClassifier(n_estimators=1000, max_depth=3, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, tree_method='hist', eval_metric='aucpr',
                      early_stopping_rounds=50, n_jobs=-1, random_state=seed)
    m.fit(X[tr], y[tr], base_margin=base[tr], eval_set=[(X[va], y[va])],
          base_margin_eval_set=[base[va]], verbose=False)
    return m

## Training gates and residual experts by fold

In [3]:
MODELS = ['Stack_global', 'Gate_pheno', 'MoE_gate', 'ResidualMoE', 'Residual_global']
oof = {t: {m: (np.full(len(win), np.nan), np.zeros(len(win), np.int8)) for m in MODELS} for t in MOE_TARGETS}
gate_weights, fold_rows = [], []
full_cols = FEATURE_SETS['full']
t_all = time.time()

for k in range(N_FOLDS):
    t_fold = time.time()
    tr, va, te = fold_masks(win, k)
    cl, dist = add_phenotype_columns(win, pheno_win, k)
    M = memberships(dist)
    X_full = win[full_cols].values
    Z_gate = win[GATE_COLS].values.astype(float)

    for t in MOE_TARGETS:
        y = win[f'y_{t}'].values
        gru, mu, sd, corr = load_gru(t, k)
        pg = sigmoid(gru_logits(gru, X_seq, seq_len, mu, sd) + corr)
        px = load_xgb('full', t, k).predict_proba(X_full)[:, 1]
        lg, lx = logit(pg), logit(px)
        res = {}

        S = np.column_stack([lg, lx])
        res['Stack_global'] = fit_gate_lr(S[va], y[va])(S)

        G = np.column_stack([lg, lx, lg[:, None] * M, lx[:, None] * M, M[:, :-1]])
        res['Gate_pheno'] = fit_gate_lr(G[va], y[va])(G)

        moe = fit_moe_gate(Z_gate[va], pg[va], px[va], y[va].astype(float), SEED + k)
        p_moe, w_moe = moe(Z_gate, pg, px)
        res['MoE_gate'] = p_moe
        for c in range(N_CLUSTERS):
            sel = te & (cl == c)
            gate_weights.append({'target': t, 'fold': k, 'cluster': c, 'w_gru_mean': float(w_moe[sel].mean()),
                                 'w_gru_sd': float(w_moe[sel].std())})

        base = lg
        g = fit_residual(X_full, base, y, tr, va, SEED + k)
        res['Residual_global'] = g.predict_proba(X_full, base_margin=base)[:, 1]
        p_res = np.empty(len(win))
        for c in np.unique(cl):
            in_c = cl == c
            if y[tr & in_c].sum() < 50 or y[va & in_c].sum() < 20:
                p_res[in_c] = res['Residual_global'][in_c]
                continue
            e = fit_residual(X_full, base, y, tr & in_c, va & in_c, SEED + k)
            p_res[in_c] = e.predict_proba(X_full[in_c], base_margin=base[in_c])[:, 1]
        res['ResidualMoE'] = p_res

        for m, score in res.items():
            thr, _ = best_threshold(y[va], score[va])
            pred = (score > thr).astype(np.int8)
            oof[t][m][0][te] = score[te]
            oof[t][m][1][te] = pred[te]
            fold_rows.append({'target': t, 'fold': k, 'model': m,
                              'auroc': roc_auc_score(y[te], score[te]),
                              'auprc': average_precision_score(y[te], score[te]),
                              'macro_f1': f1_score(y[te], pred[te], average='macro')})
    print(f'Fold {k}: {time.time() - t_fold:.0f} s')

print(f'Total: {(time.time() - t_all) / 60:.1f} min')
moe_df = win[['pid', 'cut', 'fold']].copy()
for t in MOE_TARGETS:
    for m, (score, pred) in oof[t].items():
        moe_df[f'{t}__{m}__score'] = score
        moe_df[f'{t}__{m}__pred'] = pred
moe_df.to_parquet(os.path.join(CACHE_DIR, 'oof_moe.parquet'))
pd.DataFrame(fold_rows).to_csv(os.path.join(RESULTS_DIR, 'moe_fold_metrics.csv'), index=False)
pd.DataFrame(gate_weights).to_csv(os.path.join(RESULTS_DIR, 'moe_gate_weights.csv'), index=False)

Fold 0: 66 s


Fold 1: 76 s


Fold 2: 69 s


Fold 3: 63 s


Fold 4: 62 s
Total: 5.6 min


## Summary: personalised schemes against global ones

Models from `clusterbp_v2.ipynb` on the same windows are added for comparison: `XGB_full`, `GRU`, `Blend_global_full`, `ClusterBP_full`.
Differences: patient-level bootstrap, 95% CI.

In [4]:
REF = ['XGB_full', 'GRU', 'Blend_global_full', 'ClusterBP_full']
COMPARE = [('Gate_pheno', 'Stack_global'), ('MoE_gate', 'Stack_global'), ('ResidualMoE', 'Residual_global'),
           ('MoE_gate', 'Blend_global_full'), ('ResidualMoE', 'XGB_full'), ('Stack_global', 'XGB_full')]
fold_df = pd.DataFrame(fold_rows)
tables, comps = {}, []
for t in MOE_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    names = REF + MODELS
    models = [(oof_v2[f'{t}__{m}__score'].values, oof_v2[f'{t}__{m}__pred'].values.astype(float)) for m in REF]
    models += [(oof[t][m][0], oof[t][m][1].astype(float)) for m in MODELS]
    B = boot_eval(y, win['pid'].values, models, n_boot=N_BOOT)
    np.save(os.path.join(CACHE_DIR, f'bootstrap_moe_{t}.npy'), B)
    rows = []
    for j, m in enumerate(names):
        s, p = models[j]
        rows.append({'model': m, 'AUROC': roc_auc_score(y, s), 'AUROC 95% CI': ci(B[:, j, 0]),
                     'AUPRC': average_precision_score(y, s), 'AUPRC 95% CI': ci(B[:, j, 1]),
                     'Brier': brier_score_loss(y, s),
                     'macro_F1': f1_score(y, p, average='macro'), 'macro_F1 95% CI': ci(B[:, j, 2])})
    tables[t] = pd.DataFrame(rows).set_index('model').sort_values('AUPRC', ascending=False)
    tables[t].to_csv(os.path.join(RESULTS_DIR, f'moe_metrics_{t}.csv'))
    for a, b in COMPARE:
        ia, ib = names.index(a), names.index(b)
        for mi, metric in enumerate(['AUROC', 'AUPRC', 'macro_F1']):
            d = B[:, ia, mi] - B[:, ib, mi]
            comps.append({'target': t, 'comparison': f'{a} − {b}', 'metric': metric,
                          'delta': tables[t].loc[a, metric] - tables[t].loc[b, metric], '95% CI': ci(d),
                          'p (two-sided)': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))})
comp_df = pd.DataFrame(comps)
comp_df.to_csv(os.path.join(RESULTS_DIR, 'moe_comparisons.csv'), index=False)

pd.set_option('display.width', 250)
for t, tab in tables.items():
    print(f'\n=== {t} ===')
    display(tab.round(3))


=== A ===


,AUROC,AUROC 95% CI,AUPRC,AUPRC 95% CI,Brier,macro_F1,macro_F1 95% CI
model,,,,,,,
Stack_global,0.935,[0.930; 0.939],0.545,[0.517; 0.570],0.024,0.754,[0.743; 0.764]
Gate_pheno,0.935,[0.930; 0.939],0.544,[0.516; 0.570],0.024,0.754,[0.744; 0.764]
Blend_global_full,0.934,[0.930; 0.939],0.544,[0.516; 0.569],0.024,0.754,[0.744; 0.764]
MoE_gate,0.934,[0.930; 0.939],0.543,[0.515; 0.569],0.024,0.754,[0.743; 0.764]
XGB_full,0.934,[0.929; 0.938],0.542,[0.514; 0.568],0.024,0.753,[0.743; 0.763]
ClusterBP_full,0.934,[0.929; 0.938],0.541,[0.513; 0.566],0.024,0.752,[0.742; 0.762]
Residual_global,0.931,[0.926; 0.935],0.529,[0.502; 0.554],0.025,0.748,[0.737; 0.757]
ResidualMoE,0.930,[0.925; 0.934],0.522,[0.494; 0.548],0.025,0.746,[0.736; 0.755]
GRU,0.916,[0.911; 0.922],0.475,[0.446; 0.501],0.026,0.730,[0.720; 0.741]



=== B ===


,AUROC,AUROC 95% CI,AUPRC,AUPRC 95% CI,Brier,macro_F1,macro_F1 95% CI
model,,,,,,,
MoE_gate,0.965,[0.964; 0.967],0.854,[0.845; 0.862],0.042,0.864,[0.859; 0.868]
Stack_global,0.966,[0.964; 0.967],0.854,[0.845; 0.862],0.042,0.864,[0.860; 0.868]
Gate_pheno,0.966,[0.964; 0.967],0.854,[0.845; 0.862],0.042,0.864,[0.859; 0.868]
Blend_global_full,0.965,[0.964; 0.967],0.853,[0.845; 0.862],0.042,0.864,[0.860; 0.869]
ClusterBP_full,0.965,[0.964; 0.967],0.853,[0.844; 0.862],0.042,0.864,[0.859; 0.868]
XGB_full,0.965,[0.963; 0.967],0.852,[0.843; 0.860],0.042,0.862,[0.858; 0.867]
Residual_global,0.964,[0.962; 0.965],0.849,[0.840; 0.857],0.042,0.861,[0.857; 0.866]
ResidualMoE,0.963,[0.961; 0.965],0.847,[0.838; 0.856],0.043,0.861,[0.857; 0.866]
GRU,0.957,[0.955; 0.959],0.817,[0.807; 0.828],0.046,0.851,[0.847; 0.856]



=== OLD ===


,AUROC,AUROC 95% CI,AUPRC,AUPRC 95% CI,Brier,macro_F1,macro_F1 95% CI
model,,,,,,,
Stack_global,0.931,[0.926; 0.935],0.537,[0.510; 0.562],0.026,0.749,[0.739; 0.759]
Gate_pheno,0.931,[0.926; 0.935],0.536,[0.510; 0.561],0.026,0.749,[0.738; 0.758]
MoE_gate,0.930,[0.926; 0.935],0.536,[0.509; 0.560],0.026,0.749,[0.739; 0.759]
XGB_full,0.930,[0.926; 0.934],0.535,[0.508; 0.560],0.026,0.748,[0.738; 0.758]
ClusterBP_full,0.930,[0.925; 0.934],0.534,[0.507; 0.559],0.026,0.748,[0.738; 0.757]
Blend_global_full,0.929,[0.925; 0.934],0.533,[0.506; 0.557],0.026,0.749,[0.739; 0.758]
Residual_global,0.927,[0.923; 0.932],0.523,[0.497; 0.547],0.026,0.745,[0.735; 0.754]
ResidualMoE,0.926,[0.922; 0.931],0.518,[0.491; 0.542],0.026,0.742,[0.733; 0.752]
GRU,0.913,[0.908; 0.918],0.472,[0.444; 0.497],0.028,0.727,[0.716; 0.737]


In [5]:
display(comp_df[comp_df.metric != 'AUROC'].round(4))
gw = pd.DataFrame(gate_weights)
display(gw.groupby(['target', 'cluster'])[['w_gru_mean', 'w_gru_sd']].mean().round(3))

,target,comparison,metric,delta,95% CI,p (two-sided)
1,A,Gate_pheno − Stack_global,AUPRC,-0.0003,[-0.001; 0.000],0.200
2,A,Gate_pheno − Stack_global,macro_F1,0.0003,[-0.001; 0.001],0.550
4,A,MoE_gate − Stack_global,AUPRC,-0.0011,[-0.002; -0.000],0.048
5,A,MoE_gate − Stack_global,macro_F1,0.0000,[-0.001; 0.001],1.000
7,A,ResidualMoE − Residual_global,AUPRC,-0.0070,[-0.009; -0.005],0.000
8,A,ResidualMoE − Residual_global,macro_F1,-0.0019,[-0.004; -0.000],0.034
10,A,MoE_gate − Blend_global_full,AUPRC,-0.0002,[-0.001; 0.001],0.628
11,A,MoE_gate − Blend_global_full,macro_F1,-0.0003,[-0.001; 0.001],0.462
13,A,ResidualMoE − XGB_full,AUPRC,-0.0199,[-0.024; -0.016],0.000
14,A,ResidualMoE − XGB_full,macro_F1,-0.0075,[-0.010; -0.005],0.000


w_gru_mean  w_gru_sd
target cluster                      
A      0             0.286     0.102
       1             0.250     0.100
       2             0.249     0.096
B      0             0.259     0.099
       1             0.233     0.098
       2             0.230     0.095
OLD    0             0.285     0.125
       1             0.239     0.117
       2             0.235     0.113